# Train Skip-Ad YOLO Model on Google Colab

Phase 4 of the **ad_skipper** pipeline, run **non-locally** on Colab's free GPU.

**Before you start (locally):**
1. Harvest frames: `python ad_skipper/collect_ad_frames.py --query "..." --vary-layout`
2. Group-split: `python ad_skipper/prepare_ad_dataset.py`
3. Package: `python ad_skipper/export_for_colab.py`  -> produces `ad_skipper/ad_skipper_dataset.zip`

**On Colab:** set Runtime -> Change runtime type -> GPU, then run the cells below in order.
After training, download `skip_ad_yolo.pt` and place it in `ad_skipper/models/` locally for Phase 5.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 3. Upload and extract the dataset

Run this cell, then choose `ad_skipper_dataset.zip` produced by `export_for_colab.py`.
The zip already contains a Colab-relative `data.yaml` (`path: /content/ad_skipper_dataset`).

In [ ]:
import os, zipfile
from google.colab import files

DATA_DIR = '/content/ad_skipper_dataset'
ZIP_NAME = 'ad_skipper_dataset.zip'

if not os.path.exists(ZIP_NAME):
    uploaded = files.upload()  # pick ad_skipper_dataset.zip
    ZIP_NAME = next(iter(uploaded))

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_NAME, 'r') as zf:
    zf.extractall(DATA_DIR)

print('Extracted to', DATA_DIR)
print(open(os.path.join(DATA_DIR, 'data.yaml')).read())

## 4. (Optional) Mount Google Drive instead of uploading

Skip cell 3 and use this if your zip lives in Drive. Edit the source path.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import zipfile, os
# DATA_DIR = '/content/ad_skipper_dataset'
# os.makedirs(DATA_DIR, exist_ok=True)
# with zipfile.ZipFile('/content/drive/MyDrive/ad_skipper_dataset.zip') as zf:
#     zf.extractall(DATA_DIR)

## 5. Train

`yolo11n` is light and fast for a real-time runtime loop. Adjust `epochs`/`imgsz`/`batch` as needed.

In [ ]:
from ultralytics import YOLO

BASE_MODEL = 'yolo11n.pt'
EPOCHS = 100
IMGSZ = 640
BATCH = 16

model = YOLO(BASE_MODEL)
results = model.train(
    data='/content/ad_skipper_dataset/data.yaml',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=42,
    project='/content/runs',
    name='skip_ad',
    exist_ok=True,
)
print('save_dir:', results.save_dir)

## 6. Validate

In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

## 7. Download the trained weights

Save `best.pt` as `skip_ad_yolo.pt` and place it in `ad_skipper/models/` locally for Phase 5 inference.

In [ ]:
import shutil, os
from google.colab import files

best = os.path.join(str(results.save_dir), 'weights', 'best.pt')
out = '/content/skip_ad_yolo.pt'
shutil.copy(best, out)
print('best weights:', best)
files.download(out)

# Optional: also copy to Drive
# shutil.copy(out, '/content/drive/MyDrive/skip_ad_yolo.pt')